# Análisis de Features — Importancia y Selección

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
import sys
sys.path.append('..')

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_parquet('../data/gold/dataset.parquet')
print(f'Dataset cargado: {len(df)} muestras')
print(f'Fraude: {df["is_fraud"].sum()} ({df["is_fraud"].mean()*100:.1f}%)')

In [ ]:
FEATURE_COLS = ['blur_score', 'edge_density', 'brightness', 'contrast', 
               'noise_ratio', 'symmetry_score', 'color_variance', 'ela_score',
               'moire_score', 'dct_score', 'reflection_score', 'ocr_confidence',
               'ip_risk_score', 'emulator_detected', 'tor_detected', 
               'vpn_detected', 'repeated_attempts', 'liveness_passed']

X = df[FEATURE_COLS]
y = df['is_fraud']

In [ ]:
# Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='importance', y='feature', data=importance_df, palette='viridis')
plt.title('Importancia de Features (Random Forest)')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(importance_df.head(10))

In [ ]:
# Mutual Information Scores
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='mi_score', y='feature', data=mi_df, palette='magma')
plt.title('Mutual Information Scores')
plt.xlabel('MI Score')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de señales por tipo de fraude
from src.dataset.labeler import ELA_THRESHOLD, OCR_THRESHOLD, IP_THRESHOLD, BLUR_THRESHOLD

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
signals = [
    ('ela_score', ELA_THRESHOLD, 'ELA anomalía'),
    ('ocr_confidence', OCR_THRESHOLD, 'OCR bajo'),
    ('ip_risk_score', IP_THRESHOLD, 'IP riesgo'),
    ('blur_score', BLUR_THRESHOLD, 'Blur bajo')
]

for idx, (col, threshold, label) in enumerate(signals[:4]):
    ax = axes[idx//3, idx%3]
    for ftype in df['fraud_type'].unique():
        subset = df[df['fraud_type'] == ftype]
        if col in ['ocr_confidence']:
            flagged = subset[subset[col] < threshold]
        else:
            flagged = subset[subset[col] > threshold] if col != 'blur_score' else subset[subset[col] < threshold]
        ax.bar(ftype, len(flagged)/len(subset)*100 if len(subset) > 0 else 0, label=ftype, alpha=0.7)
    ax.set_title(f'% muestras con {label}')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()